In [1]:
!pip install -q scikit-learn

In [2]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_from_disk, concatenate_datasets, Audio
import glob
from datasets import disable_caching
from torch.utils.data import DataLoader
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    roc_curve,
    auc,
    roc_auc_score,
)

from torchvision.models import resnet18

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd /content/drive/MyDrive/Speaker-Recognition-using-ResNet-Embeddings

/content/drive/MyDrive/Speaker-Recognition-using-ResNet-Embeddings


## Load Data and DataLoader (dev and test set)

In [4]:
disable_caching()

# loading data

chunk_paths = sorted(glob.glob("speaker_chunks/chunk_*"))

chunks = []
for path in chunk_paths:
    ds = load_from_disk(path)
    ds = ds.cast_column("audio_path", Audio(sampling_rate=16000))
    chunks.append(ds)

full_dataset = concatenate_datasets(chunks)

In [5]:
# fixed held-out test set
split = full_dataset.train_test_split(test_size=0.1, seed=42)
trainval_dataset = split["train"]   # for CV
test_dataset = split["test"]  # fixed

# get the labels for stratification
labels = np.array(trainval_dataset["label"])

# 9-way stratified split, take 3 folds
skf = StratifiedKFold(n_splits=9, shuffle=True, random_state=42)
fold_indices = list(skf.split(np.zeros(len(labels)), labels))

n_folds = 3
folds = [] #collect the 3 folds
for i in range(n_folds):
    train_idx, dev_idx = fold_indices[i]
    train_dataset = trainval_dataset.select(train_idx.tolist())
    dev_dataset = trainval_dataset.select(dev_idx.tolist())
    folds.append((train_dataset, dev_dataset)) #collect the 3 folds
    print(f"Fold {i}: train={len(train_dataset)}, dev={len(dev_dataset)}, test={len(test_dataset)}")

Fold 0: train=20000, dev=2500, test=2500
Fold 1: train=20000, dev=2500, test=2500
Fold 2: train=20000, dev=2500, test=2500


In [7]:
dataset_dev_folds = []

#loop for 3 folds


from dataset import Dataset_Builder

config = {"shortest_duration": 4.0}

for fold in folds:
  dev_builder = Dataset_Builder(fold[1], **config)
  dev_builder.filter()
  dev_builder.preprocess()
  dataset_dev_folds.append(dev_builder.get_dataset())

test_builder = Dataset_Builder(test_dataset, **config)
test_builder.filter()
test_builder.preprocess()
test_dataset = test_builder.get_dataset()

Duration of cropped log-mel spectograms (input):  4.0  seconds


Filter:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2407 [00:00<?, ? examples/s]

Duration of cropped log-mel spectograms (input):  4.0  seconds


Filter:   0%|          | 0/2500 [00:00<?, ? examples/s]

...

Filter:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2400 [00:00<?, ? examples/s]

Duration of cropped log-mel spectograms (input):  4.0  seconds


Filter:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2401 [00:00<?, ? examples/s]

In [8]:
def collate_fn(batch):
    log_mels = torch.stack([b["log_mel"] for b in batch])
    labels = torch.tensor([b["label"] for b in batch])
    return (log_mels, labels)

In [9]:
dataloader_dev_folds = []

#loop for folds
for dev_dataset in dataset_dev_folds:
    dataloader_dev = DataLoader(
        dev_dataset,
        batch_size=32,
        shuffle=False,
        collate_fn=collate_fn,
        pin_memory=True
    )
    dataloader_dev_folds.append(dataloader_dev)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    collate_fn=collate_fn
)

## Accuracy Evaluation for folds

In [21]:
def evaluate_acc(model, dataloader, device):
    model.eval()

    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for inputs, labels in tqdm(dataloader, total=len(dataloader)):
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)

            batch_size = labels.size(0)

            total_correct += (outputs.argmax(dim=1) == labels).sum().item()
            total_samples += batch_size

    accuracy = total_correct / total_samples

    return accuracy

def fold_eval_acc(checkpoint_path,checkpoint_name,num_classes):
    fold_accs = []
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    for fold in range(3):
        model = resnet18(weights=None)
        model.fc = torch.nn.Linear(model.fc.in_features, num_classes)
        model.load_state_dict(torch.load(f"{checkpoint_path}/{checkpoint_name}{fold}.pth"))
        model.to(device)

        dev_acc = evaluate_acc(model, dataloader_dev_folds[fold], device)
        test_acc = evaluate_acc(model, test_loader, device)

        fold_accs.append((dev_acc, test_acc))

    mean_dev_acc = np.mean([acc[0] for acc in fold_accs])
    mean_test_acc = np.mean([acc[1] for acc in fold_accs])
    std_dev_acc = np.std([acc[0] for acc in fold_accs])
    std_test_acc = np.std([acc[1] for acc in fold_accs])

    print(
        f"{checkpoint_path}"
    )

    print(
    f"dev Accuracy: {mean_dev_acc * 100:.2f}% "
    f"± {std_dev_acc * 100:.2f}%"
    )
    print(
    f"test Accuracy: {mean_test_acc * 100:.2f}% "
    f"± {std_test_acc * 100:.2f}%"
    )
    return fold_accs

In [22]:
fold_accs_CE_6epoch = fold_eval_acc("speaker_chunks/resnet_cross_val_6e_4s","best_speaker_resnet_fold",50)
fold_accs_EC_3epoch = fold_eval_acc("speaker_chunks/resnet_cross_val_3e_4s","4sec_3epoch_speaker_resnet_fold",50)
fold_accs_probing = fold_eval_acc("speaker_chunks/resnet_cross_val_6e_4s_probing", "4sec_6epoch_probing_resnet_fold", 50)


100%|██████████| 76/76 [03:53<00:00,  3.07s/it]


speaker_chunks/resnet_cross_val_6e_4s
dev Accuracy: 94.44% ± 0.79%
test Accuracy: 95.06% ± 0.17%


100%|██████████| 76/76 [03:46<00:00,  2.98s/it]


speaker_chunks/resnet_cross_val_3e_4s
dev Accuracy: 92.70% ± 0.53%
test Accuracy: 93.54% ± 0.15%


100%|██████████| 76/76 [03:50<00:00,  3.03s/it]

speaker_chunks/resnet_cross_val_6e_4s_probing
dev Accuracy: 21.54% ± 0.92%
test Accuracy: 22.23% ± 0.68%
